# Análise dos resultados — melhor configuração de cada algoritmo

Roda **localmente**, sem treinar nada: lê todos os `metadata.json` de
`results/avaliacao_5_particoes/` (rodadas 3–8; a pasta `vazamento/` fica de
fora). Para cada algoritmo, mostra a configuração de maior KS médio no teste
nas 5 partições, com seus hiperparâmetros, protocolo de treino e quantas
configurações daquele algoritmo foram testadas.

Para incluir a rodada 8, traga a branch `results/churn-rodada8-paciencia`
(`git fetch origin results/churn-rodada8-paciencia && git merge FETCH_HEAD`) e rode de novo.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from churn_telecom import relatorio

pd.set_option("display.max_colwidth", 200)
longo = relatorio.carregar_5_particoes(PROJECT_ROOT / "results")
print(f"{longo.groupby(['rodada', 'modelo']).ngroups} configuracoes, {len(longo)} modelos treinados")

## Tabela: melhor de cada algoritmo

In [ ]:
melhores = relatorio.gerar([PROJECT_ROOT / "results"], out_dir=PROJECT_ROOT)
tabela = melhores[["familia", "ks_mean", "ks_std", "auroc_mean", "acuracia_limiar_validacao_mean", "rodada",
                   "modelo", "treino", "codificacao", "transformacao", "iteracoes_particao0",
                   "configuracoes_testadas", "hiperparametros"]].rename(columns={
    "familia": "Algoritmo", "ks_mean": "KS", "ks_std": "KS desvio", "auroc_mean": "AUROC",
    "acuracia_limiar_validacao_mean": "Acurácia (limiar val.)", "iteracoes_particao0": "épocas/árvores (p0)",
    "configuracoes_testadas": "configs testadas"})
tabela.style.format({"KS": "{:.3f}", "KS desvio": "{:.3f}", "AUROC": "{:.3f}",
                     "Acurácia (limiar val.)": "{:.1%}"}, na_rep="-").hide(axis="index")

## Figura: KS do melhor de cada algoritmo

Pontos = média nas 5 partições; barras = ± 1 desvio.

In [ ]:
from IPython.display import Image
Image(PROJECT_ROOT / "reports" / "figures" / "melhores_por_algoritmo_ks.png")

## Rodada 8 — efeito da paciência, da busca e das transformações

Só aparece depois de trazer a branch da rodada 8.

In [ ]:
t = PROJECT_ROOT / "reports" / "tables"
if (t / "rodada8_comparacoes_pareadas.csv").exists():
    display(pd.read_csv(t / "rodada8_resumo.csv").head(25))
    display(pd.read_csv(t / "rodada8_comparacoes_pareadas.csv"))
else:
    print("rodada 8 ainda nao disponivel localmente")

### Paciência vs. épocas/árvores fixas (rodada 3 → rodada 8)

Mesmas 5 partições (sementes 0–4), então a diferença é pareada. Atenção: a
rodada 8 também passou a usar a validação sem cópias para a parada, então o
efeito medido é "paciência + protocolo novo".

In [ ]:
r3 = longo[longo.rodada == "rodada3"]
r8 = longo[longo.rodada == "rodada8"]
linhas = []
for alg in ("mlp", "gb", "xgb", "stab", "tabkan", "kan"):
    for antigo, novo in ((f"{alg}_padrao", f"{alg}_padrao_paciencia"), (f"{alg}_optuna", f"{alg}_optuna_paciencia")):
        a = r3[r3.modelo == antigo].set_index("particao").ks
        b = r8[r8.modelo == novo].set_index("particao").ks
        d = (b - a).dropna()
        if len(d) == 5:
            linhas.append({"antes": antigo, "depois": novo, "ks_antes": a.mean(), "ks_depois": b.mean(),
                           "delta": d.mean(), "delta_desvio": d.std()})
pd.DataFrame(linhas) if linhas else print("rodada 8 ainda nao disponivel localmente")